<a href="https://colab.research.google.com/github/crzed/backup-psql/blob/main/Music_Source_Separation_Training_(Colab_Inference)_CustomModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab inference for ZFTurbo's [Music-Source-Separation-Training](https://github.com/ZFTurbo/Music-Source-Separation-Training/)


<font size=1>*made by jarredou & deton</font>  
[![ko-fi](https://ko-fi.com/img/githubbutton_sm.svg)](https://ko-fi.com/Q5Q811R5YI)  

In [1]:
#@markdown #Gdrive connection
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
import base64
#@markdown # Install

%cd /content
!git clone -b colab-inference-iterative-method https://github.com/Qupci/Music-Source-Separation-Training

#requirements fix by santilli_
req_text = """
mutagen==1.47.0
ml_collections==1.1.0
numpy>=1.26.0
pandas
scipy
tqdm
segmentation_models_pytorch==0.3.3
timm
audiomentations
pedalboard
omegaconf
beartype
rotary_embedding_torch==0.3.5
einops
# librosa==0.11.0
demucs #==4.0.0
# transformers==4.35.0
torchmetrics==0.11.4
spafe==0.3.2
protobuf
torch_audiomentations
asteroid==0.7.0
auraloss
torchseg
"""

with open("Music-Source-Separation-Training/requirements.txt", "w") as f:
    f.write(req_text)

!mkdir '/content/Music-Source-Separation-Training/ckpts'

print('Installing the dependencies... This will take a few minutes')
!pip install -r 'Music-Source-Separation-Training/requirements.txt' &> /dev/null

print('Installation is done !')

/content
Cloning into 'Music-Source-Separation-Training'...
remote: Enumerating objects: 2306, done.
remote: Counting objects: 100% (919/919), done.
remote: Compressing objects: 100% (167/167), done.
remote: Total 2306 (delta 827), reused 752 (delta 752), pack-reused 1387 (from 2)
Receiving objects: 100% (2306/2306), 3.49 MiB | 8.21 MiB/s, done.
Resolving deltas: 100% (1460/1460), done.
Installing the dependencies... This will take a few minutes
Installation is done !


In [9]:
%cd '/content/Music-Source-Separation-Training/'
import os
import yaml
from urllib.parse import quote
import urllib.request

class IndentDumper(yaml.Dumper):
    def increase_indent(self, flow=False, indentless=False):
        return super(IndentDumper, self).increase_indent(flow, False)


def tuple_constructor(loader, node):
    # Load the sequence of values from the YAML node
    values = loader.construct_sequence(node)
    # Return a tuple constructed from the sequence
    return tuple(values)

# Register the constructor with PyYAML
yaml.SafeLoader.add_constructor('tag:yaml.org,2002:python/tuple',
tuple_constructor)



def conf_edit(config_path, chunk_size, overlap):
    with open(config_path, 'r') as f:
        data = yaml.load(f, Loader=yaml.SafeLoader)

    # handle cases where 'use_amp' is missing from config:
    if 'training' in data and 'use_amp' not in data['training']:
      data['training']['use_amp'] = True

    if 'audio' in data:
        data['audio']['chunk_size'] = chunk_size
    else:
        data['audio'] = {'chunk_size': chunk_size}

    if 'inference' in data:
        data['inference']['num_overlap'] = overlap
        if 'batch_size' not in data['inference'] or data['inference'].get('batch_size') == 1:
            data['inference']['batch_size'] = 4
    else:
        data['inference'] = {'num_overlap': overlap, 'batch_size': 4}

    print("Using custom overlap and chunk_size values for roformer model:")
    print(f"overlap = {data['inference']['num_overlap']}")
    print(f"chunk_size = {data['audio']['chunk_size']}")
    print(f"batch_size = {data['inference'].get('batch_size')}")


    with open(config_path, 'w') as f:
        yaml.dump(data, f, default_flow_style=False, sort_keys=False, Dumper=IndentDumper, allow_unicode=True)

def download_file(url):
    # Encode the URL to handle spaces and special characters
    encoded_url = quote(url, safe=':/')

    path = 'ckpts'
    os.makedirs(path, exist_ok=True)
    filename = os.path.basename(encoded_url)
    file_path = os.path.join(path, filename)

    if os.path.exists(file_path):
        print(f"File '{filename}' already exists at '{path}'.")
        return

    try:
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(encoded_url, file_path)
        print(f"File '{filename}' downloaded successfully")
    except Exception as e:
        print(f"Error downloading file '{filename}' from '{url}': {e}")





#@markdown # Separation
#@markdown #### Model config:
config_url = 'https://raw.githubusercontent.com/KimberleyJensen/Mel-Band-Roformer-Vocal-Model/main/configs/config_vocals_mel_band_roformer.yaml' #@param {type:"string"}
ckpt_url = 'https://huggingface.co/KimberleyJSN/melbandroformer/resolve/main/MelBandRoformer.ckpt' #@param {type:"string"}
model_type = 'mel_band_roformer' #@param ['mdx23c','bs_roformer', 'mel_band_roformer', 'bandit', 'bandit_v2', 'scnet', 'apollo', 'htdemucs', 'segm_models', 'torchseg', 'bs_mamba2']
#@markdown ---
#@markdown #### Separation config:
input_folder = '/content/drive/MyDrive/input' #@param {type:"string"}
output_folder = '/content/drive/MyDrive/output' #@param {type:"string"}
extract_instrumental = True #@param {type:"boolean"}
export_format = 'flac PCM_16' #@param ['wav FLOAT', 'flac PCM_16', 'flac PCM_24']
use_tta = True #@param {type:"boolean"}
#@markdown ---
#@markdown *Roformers custom config:*
overlap = 2 #@param {type:"slider", min:2, max:40, step:1}
chunk_size = "485100" #@param [352800, 485100] {allow-input: true}

if export_format.startswith('flac'):
    flac_file = True
    pcm_type = export_format.split(' ')[1]
else:
    flac_file = False
    pcm_type = None


if config_url != '' and ckpt_url != '':
    config_filename = os.path.basename(config_url)
    ckpt_filename = os.path.basename(ckpt_url)
    print(config_filename, ckpt_filename)
    config_path = f'ckpts/{config_filename}'
    start_check_point = f'ckpts/{ckpt_filename}'
    # Always force overwrite config to ensure updates to it are applied correctly
    if os.path.exists(config_path):
        os.remove(config_path)
    download_file(config_url)
    download_file(ckpt_url)
    if "roformer" in model_type:
      conf_edit(config_path, int(chunk_size), overlap)

!python inference.py \
    --model_type {model_type} \
    --config_path '{config_path}' \
    --start_check_point '{start_check_point}' \
    --input_folder '{input_folder}' \
    --store_dir '{output_folder}' \
    {('--extract_instrumental' if extract_instrumental else '')} \
    {('--flac_file' if flac_file else '')} \
    {('--use_tta' if use_tta else '')} \
    {('--pcm_type ' + pcm_type if pcm_type else '')}

/content/Music-Source-Separation-Training
config_vocals_mel_band_roformer.yaml MelBandRoformer.ckpt
File 'config_vocals_mel_band_roformer.yaml' downloaded successfully
File 'MelBandRoformer.ckpt' already exists at 'ckpts'.
Using custom overlap and chunk_size values for roformer model:
overlap = 2
chunk_size = 485100
batch_size = 4
CUDA is available, use --force_cpu to disable it.
Using device:  cuda:0
GPU Compute Capability below 8.0, using math or mem efficient attention if input tensor is on cuda
Start from checkpoint: ckpts/MelBandRoformer.ckpt
Instruments: ['vocals', 'other']
Model load time: 3.76 sec
Total files found: 1 Use sample rate: 44100
Starting processing track:  /content/drive/MyDrive/input/vecher.mp3
Elapsed time: 21.27 sec


1* Links to the models pasted above should look as below:<br><br>
`https://huggingface.co/becruily/mel-band-roformer-instrumental/resolve/main/config_instrumental_becruily.yaml` <br><br>
`https://huggingface.co/becruily/mel-band-roformer-instrumental/resolve/main/mel_band_roformer_instrumental_becruily.ckpt`
<br><br>
To possess such links, you need to go to model card, open the model file from the file list, and click "Copy download link" and paste above.<br><br>
These links shouldn't end with `download=true`
<br><br>
________________________________________________________
2* Links from GitHub should look as following:<br><br>
(config from repository) <br>
`https://raw.githubusercontent.com/ZFTurbo/Music-Source-Separation-Training/main/configs/KimberleyJensen/config_vocals_mel_band_roformer_kj.yaml`
<br><br>
To possess such link you need to locate the file in the repository and when its content is shown, click RAW and copy the destination address from your browser.
<br><br>
(from releases page)<br>
`https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v.1.0.4/model_mel_band_roformer_crowd.yaml`<br><br>
`https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v.1.0.4/mel_band_roformer_crowd_aufr33_viperx_sdr_8.7144.ckpt<br><br>
Just copy the links from there and paste here

_________________________________________________________________

**INST-Mel-Roformer v1/1x/2** has switched output file names - files labelled as vocals are instrumentals (if you uncheck extract_instrumentals for v1e model, only one stem caled "other" will be rendered, and it will be instrumental.<br><br>
**TTA** - results in longer separation time, "it gives a little better SDR score but hard to tell if it's really audible in most cases". <br> it “means "test time augmentation", (...) it will do 3 passes on the audio file instead of 1. 1 pass with be with original audio. 1 will be with inverted stereo (L becomes R, R become L). 1 will be with phase inverted and then results are averaged for final output. ” - jarredou
<br><br>
**Overlap** - higher means longer separation time. 4 is already balanced value, 2 is fast and some people still won't notice any difference. Normally there's not point going over 8.<br><br>
**Chunk_size** - most models use the default 485100 (besides e.g. Deux - 573300, Beta6X - 529200, 6 stems SW - 588800 [882000 will probably crash on free T4], and Amane's 4 stems Large - 661500) but some people occasionally use also 112455 to get different results. Usually higher inference chunk_size than the training chunk_size gives better SDR. Most trainers are aware of it and provide already correct value in the yaml.

<br><br>
If your separation can't start and "Total files found: 0" is shown, be aware that: <br>1) Input must be a path to a folder containing audio files, not direct path to an audio file<br> 2) The Colab is case aware - e.g. call your folder "input" not "Input".<br> 3) Check if your Google Drive mounting was executed correctly. Open file manager on the left to check if your drive folder is not empty. If it's the case, force remount with the following line:

In [ ]:
drive.mount("/content/drive", force_remount=True)